In [1]:
!pip uninstall -y torchao
!pip install pandas numpy scikit-learn transformers bitsandbytes accelerate peft torch torchvision

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")

In [5]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/hate_train.csv")
df.head()

,sentence,label
0,Dla mnie faworytem do tytułu będzie Cracovia. ...,0
1,@anonymized_account @anonymized_account Brawo ...,0
2,"@anonymized_account @anonymized_account Super,...",0
3,@anonymized_account @anonymized_account Musi. ...,0
4,"Odrzut natychmiastowy, kwaśna mina, mam problem",0


In [6]:
print(df["label"].value_counts())

label
0    9190
1     851
Name: count, dtype: int64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df["sentence"].to_list(), df["label"].to_list(), test_size=0.10, random_state=42, stratify=df["label"])
X_train, X_eval, y_train, y_eval = train_test_split(X_train, y_train, test_size=1/9, random_state=42, stratify=y_train)
X_train, y_train

(['@anonymized_account @anonymized_account Wspaniały. Ale takiego Więckiewicza się można było spodziewać. Takiego Frycza - bynajmniej! Tu leży jego przewaga.',
  '@anonymized_account Tylko czemu link do tekstu z lutego? :P',
  '@anonymized_account @anonymized_account @anonymized_account Np to że Misiek i Zielak zainwestowali miliony w klub 😂',
  '@anonymized_account zjeby mówią że nie chodziło fizycznie o gotówkę, kurwa zero godności',
  '@anonymized_account A nastepni będą tacy jak Terlikowski spędzający czas w kolejce do psychiatry.',
  '@anonymized_account @anonymized_account @anonymized_account @anonymized_account @anonymized_account Po cholerę ci pytania? Przecież dostaniesz z PiSu na kartce.',
  'RT @anonymized_account wyniki sondy były do przewidzenia Polacy chcieli i chcą nadal reform sądów ! #bezretuszu',
  '@anonymized_account @anonymized_account Euro 2016 pokazało trochę co innego.',
  '@anonymized_account Ale skoro lubi śpiewać w samochodzie, więc zaśpiewa, to dlaczego z ni

In [8]:
class CustomTextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [9]:
# model_name = "distilbert-base-uncased"
model_name = "sdadas/polish-roberta-base-v2"
# model_name = "allegro/herbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# q_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True,
    output_attentions=False, output_hidden_states=False,
    # quantization_config=q_config,
)
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: sdadas/polish-roberta-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
datasets = {
    "train": CustomTextClassificationDataset(texts=X_train, labels=y_train, tokenizer=tokenizer),
    "test": CustomTextClassificationDataset(texts=X_test, labels=y_test, tokenizer=tokenizer),
    "eval": CustomTextClassificationDataset(texts=X_eval, labels=y_eval, tokenizer=tokenizer)
}

In [11]:
# !inputs must be weighted as negative sentiment class (1) is undersampled

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # weighted CE
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

print(model)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/results",
    num_train_epochs=20,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir="/content/drive/MyDrive/Colab Notebooks//logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=2e-4,
    fp16=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50001, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): Module

In [12]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.494829,0.710583,0.914343
2,0.866633,0.575656,0.915339
3,0.480213,0.412153,0.928287
4,0.078481,0.616372,0.918327
5,0.001927,0.598317,0.922311


TrainOutput(global_step=20080, training_loss=0.5105027255476141, metrics={'train_runtime': 1201.6058, 'train_samples_per_second': 133.688, 'train_steps_per_second': 66.844, 'total_flos': 2678089861201920.0, 'train_loss': 0.5105027255476141, 'epoch': 5.0})

In [13]:
eval_dataloader = DataLoader(datasets["test"], batch_size=8, shuffle=False, drop_last=False, num_workers=4)

all_predictions = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in eval_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_predictions)
print(f"Model Accuracy: {accuracy:.4f}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Model Accuracy: 0.9234


In [14]:
with open("/content/drive/MyDrive/Colab Notebooks/hate_test_data.txt", "r", encoding="utf-8") as f:
    test_data = f.read().splitlines()

all_predictions = []

model.eval()
with torch.no_grad():
  for sentence in test_data:
      encoding = tokenizer(
          sentence,
          truncation=True,
          padding="max_length",
          max_length=128,
          return_tensors="pt",
      )

      input_ids = encoding["input_ids"].to(device)
      attention_mask = encoding["attention_mask"].to(device)

      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      predictions = torch.argmax(outputs.logits, dim=-1)

      all_predictions.extend(predictions.cpu().numpy())
print(all_predictions)

all_predictions = np.array(all_predictions, dtype=np.int32)

np.savetxt("/content/drive/MyDrive/Colab Notebooks/pred.csv", all_predictions, fmt='%d')

[np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0)

In [15]:
print(np.unique_counts(all_predictions))

UniqueCountsResult(values=array([0, 1], dtype=int32), counts=array([948,  52]))
